The following function computes $s_n[s_\mu]$ using the recursive formula:
## $$n h_n[s_\mu] = \sum_{k=1}^n p_k[s_\mu] h_{n-k}[s_\mu]$$

In [27]:
#the following is the monomial expansion of s_mu in d-variables
@cached_function
def mc_s_mu(mu,d):
    return list(s(mu).expand(d).monomial_coefficients().items())

# defaultdict is useful for adding values adict[key] += c means that if key is already a key in adict
# it will add c to the current value, if not, it makes c the initial value.
from collections import defaultdict

# this is a function that computes a dictionary for the terms in s_n[s_mu]
# do it for d variables
# this has worked really well for computing s_n[s_m] for a fixed m and for enn=0,1,2,3,...
# because I want the output in x1,x2,...,xd plus an extra variable z the length
# of the tuples are going to be d+1
def is_weakly_decreasing(tup):
    """
    test if a tuple is weakly decreasing
    """
    return all(tup[i]>=tup[i+1] for i in range(len(tup)-1))
@cached_function
def S_d(d):
    """
    The symmetric group with a sign
    """
    return [(tuple(v-1 for v in p),p.sign()) for p in Permutations(d)]
def sn_smu(enn, mu, d):
    """
    compute the terms in s_n[s_mu] of length at most d by calling hn_smu
    (this is a wrapper function so that lists are accepted in a cached function)
    """
    return hn_smu(enn, Partition(mu), d)
@cached_function
def hn_smu(enn, mu, d):
    """
    compute h_n[s_mu] using the recursive formula
    n h_n[s_mu] = sum_{k=1}^n p_k[s_mu] h_{n-k}[s_mu]
    """
    if enn==0:
        return { (0,)*d : 1 }
    else:
        out = defaultdict(int)
        for k in range(1,enn+1):
            for (w,ccc) in mc_s_mu(mu,d):
                for (v,c) in hn_smu(enn-k,mu,d).items():
                    for (p,cc) in S_d(d):
                        wv = tuple(k*w[i]+v[p[i]]+i-p[i] for i in range(d))
                        if is_weakly_decreasing(wv):
                            out[wv]+=c*cc*ccc
        return dict({v:c//enn for (v,c) in out.items() if c!=0})

# WHAT WE DID ON May 28 STARTS HERE:

The following function computes s_n[s_4] using the recursive formula:
## $$n h_n[s_4] = \sum_{k=1}^n p_k[s_4] h_{n-k}[s_4]$$

In [10]:
SymmetricFunctions(QQ).inject_shorthands(verbose=False)
vl=var('x1,x2,z')
nv = 2 # number of variables
mc4=list(s[4].expand(nv,alphabet=list(vl)[:nv]).monomial_coefficients().keys())
# mc4 are all of the monomials in s_4(x1,x2)

In [11]:
mc4

[(4, 0), (3, 1), (2, 2), (1, 3), (0, 4)]

In [12]:
s[4].expand(2)

x0^4 + x0^3*x1 + x0^2*x1^2 + x0*x1^3 + x1^4

In [14]:
S2 = [((p[0]-1,p[1]-1),p.sign()) for p in Permutations(2)]
S2

[((0, 1), 1), ((1, 0), -1)]

In [41]:
from collections import defaultdict
@cached_function
def hn_s4(enn):
    if enn==0:
        return { (0,0,0) : 1 }
    else:
        out = defaultdict(int)
        for k in range(1,enn+1):
            for w in mc4:
                for (v,c) in hn_s4(enn-k).items():
                    for (p,cc) in S2:
                        wv = (k*w[0]+v[p[0]]-p[0],k*w[1]+v[p[1]]+1-p[1])
                        if wv[0]>=wv[1]:
                            out[wv+(0,)]+=c*cc
        return dict({v:c//enn for (v,c) in out.items() if c!=0})

In [42]:
hn_s4(5)

{(20, 0, 0): 1,
 (18, 2, 0): 1,
 (16, 4, 0): 2,
 (17, 3, 0): 1,
 (14, 6, 0): 2,
 (12, 8, 0): 2,
 (15, 5, 0): 1,
 (13, 7, 0): 1,
 (10, 10, 0): 1}

In [45]:
sum(c*s(la) for (la,c) in s[5](s[4]) if len(la)<=2)

s[10, 10] + 2*s[12, 8] + s[13, 7] + 2*s[14, 6] + s[15, 5] + 2*s[16, 4] + s[17, 3] + s[18, 2] + s[20]

In [61]:
conj_den = (1-z*x1**2*x2**2)*(1-z**6*x1**12*x2**12)*(1-z*x1**3*x2)*(1-z*x1**4)

In [62]:
BR = QQ['x1,x2,z']
et = sage.rings.polynomial.polydict.ETuple
@cached_function
def conj_den_4():
    return BR(expand(
        conj_den
        )).dict().items()#
@cached_function
def den_coeff(d):
    return BR._from_dict({ et(list(t)[:-1]+[0]) : c for (t,c) in conj_den_4() if list(t)[-1]==d})
def calc_num(d):
    return sum(den_coeff(d-r)*BR(hn_s4(r)) for r in range(d+1))
def diff_tup(p,q):
    return tuple([a-b for (a,b) in zip(q,p)])

In [63]:
out={}
for d in range(0,100):
    CC = calc_num(d)
    CCC = sorted(list(CC),key=lambda m: list(m[1].leading_item()[0])[::-1],reverse=True)
    #CCC=list(CC)
    if CC:
        out[d]=CCC[0][1].leading_item()[0]
        print(d,out[d], len(list(CC)),CCC[0])
        delta = 1
        if d-delta in out:
            print(diff_tup(out[d-delta],out[d]))
        print("*************")
    else:
        print(d, "####" )

0 (0, 0, 0) 1 (1, 1)
*************
1 (2, 2, 0) 2 (-1, x1^2*x2^2)
(2, 2, 0)
*************
2 (4, 4, 0) 3 (1, x1^4*x2^4)
(2, 2, 0)
*************
3 (7, 5, 0) 2 (-1, x1^7*x2^5)
(3, 1, 0)
*************
4 (10, 6, 0) 1 (1, x1^10*x2^6)
(3, 1, 0)
*************
5 ####
6 ####
7 ####
8 ####
9 ####
10 ####
11 ####
12 ####
13 ####
14 ####
15 ####
16 ####
17 ####
18 ####
19 ####
20 ####
21 ####
22 ####
23 ####
24 ####
25 ####
26 ####
27 ####
28 ####
29 ####
30 ####
31 ####
32 ####
33 ####
34 ####
35 ####
36 ####
37 ####
38 ####
39 ####
40 ####
41 ####
42 ####
43 ####
44 ####
45 ####
46 ####
47 ####
48 ####
49 ####
50 ####
51 ####
52 ####
53 ####
54 ####
55 ####
56 ####
57 ####
58 ####
59 ####
60 ####
61 ####
62 ####
63 ####
64 ####
65 ####
66 ####
67 ####
68 ####
69 ####
70 ####
71 ####
72 ####
73 ####
74 ####
75 ####
76 ####
77 ####
78 ####
79 ####
80 ####
81 ####
82 ####
83 ####
84 ####
85 ####
86 ####
87 ####
88 ####
89 ####
90 ####
91 ####
92 ####
93 ####
94 ####
95 ####
96 ####
97 ####
98 ####
99

In [71]:
out={}
for d in range(0,100):
    CC = calc_num(d)
    print(z**d*CC)

1
-(x1^3*x2 + x1^2*x2^2)*z
(x1^6*x2^2 + x1^5*x2^3 + x1^4*x2^4)*z^2
-(x1^8*x2^4 + x1^7*x2^5)*z^3
x1^10*x2^6*z^4
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0


In [73]:
B4_2vars=(1-(x1^3*x2 + x1^2*x2^2)*z+(x1^6*x2^2 + x1^5*x2^3 + x1^4*x2^4)*z^2-(x1^8*x2^4 + x1^7*x2^5)*z^3+x1^10*x2^6*z^4)/conj_den

In [74]:
B4_2vars

(x1^10*x2^6*z^4 - (x1^8*x2^4 + x1^7*x2^5)*z^3 + (x1^6*x2^2 + x1^5*x2^3 + x1^4*x2^4)*z^2 - (x1^3*x2 + x1^2*x2^2)*z + 1)/((x1^12*x2^12*z^6 - 1)*(x1^4*z - 1)*(x1^3*x2*z - 1)*(x1^2*x2^2*z - 1))

In [84]:
expand(taylor(B4_2vars,z,0,5).coefficient(z,5))

x1^20 + x1^18*x2^2 + x1^17*x2^3 + 2*x1^16*x2^4 + x1^15*x2^5 + 2*x1^14*x2^6 + x1^13*x2^7 + 2*x1^12*x2^8 + x1^10*x2^10

In [80]:
hn_s4(5)

{(20, 0, 0): 1,
 (18, 2, 0): 1,
 (16, 4, 0): 2,
 (17, 3, 0): 1,
 (14, 6, 0): 2,
 (12, 8, 0): 2,
 (15, 5, 0): 1,
 (13, 7, 0): 1,
 (10, 10, 0): 1}